# buy_stock — human-in-the-loop confirmation notebook

Interactive scratchpad for exercising `buy_stock`, the one order-placing
tool in this project, and its double-confirmation gate.

The gate is enforced by two `langgraph.types.interrupt()` calls inside the
tool (`app/tools.py`) — not by prompt wording. That means it can be
verified two ways, and this notebook does both:

1. **Section 1** drives the real LangGraph interrupt/resume mechanics
   directly against a minimal tool graph — no LLM, no API key, no cost.
   This is the same technique `tests/test_tools.py` uses.
2. **Section 2 onward** drives the full agent (`app/agent.py`) through the
   actual configured LLM, exactly as the CLI or FastAPI endpoint would,
   to confirm `ask()`'s resume-detection plumbing works end-to-end.

Run cells top to bottom. If you edit `app/*.py`, re-run the import cell
(autoreload picks up changes automatically).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the project root (this notebook's directory) is importable as `app.*`
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.config import settings

print(f"LLM provider: {settings.llm_provider}")
print(f"Using mock IBKR data: {settings.use_mock_ibkr}")

LLM provider: groq
Using mock IBKR data: True


## 1. Graph-level test — no LLM involved

Builds a minimal single-node graph around `buy_stock` and drives it with a
hand-built tool call, the same way `tests/test_tools.py` does. This proves
the confirmation gate is a property of the graph itself, not of the model's
behavior — nothing here can talk the tool into skipping a step.

In [2]:
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

from app.tools import buy_stock


def build_tool_graph():
    graph = StateGraph(MessagesState)
    graph.add_node("tools", ToolNode([buy_stock]))
    graph.set_entry_point("tools")
    graph.set_finish_point("tools")
    return graph.compile(checkpointer=MemorySaver())


def buy_call(symbol: str, quantity: float) -> dict:
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {"name": "buy_stock", "args": {"symbol": symbol, "quantity": quantity}, "id": "call1"}
                ],
            )
        ]
    }


graph = build_tool_graph()
config = {"configurable": {"thread_id": "notebook-graph-1"}}

result = graph.invoke(buy_call("AAPL", 10), config=config)
print("Paused for confirmation #1:", "__interrupt__" in result)
print(result["__interrupt__"][0].value["message"])

Paused for confirmation #1: True
Confirm order: BUY 10.0 AAPL @ ~$194.10 (estimated cost $1,941.00). Reply 'yes' to continue or 'no' to cancel.


In [3]:
# The LLM is never re-invoked between these two pauses — only a real
# Command(resume=...) call from outside the graph can move it forward.
result = graph.invoke(Command(resume="yes"), config=config)
print("Paused for confirmation #2:", "__interrupt__" in result)
print(result["__interrupt__"][0].value["message"])

Paused for confirmation #2: True
Final confirmation — this will submit a real order: BUY 10.0 AAPL @ ~$194.10 (estimated cost $1,941.00). This cannot be undone once submitted. Reply 'yes' to submit or 'no' to cancel.


In [4]:
result = graph.invoke(Command(resume="yes"), config=config)
print(result["messages"][-1].content)

Order submitted: BUY 10.0 AAPL @ ~$194.10 (order id 2001, status Submitted).


### 1a. Aborting at either step places nothing

A "no" (or anything that isn't a clear yes) at *either* confirmation ends
the flow without ever calling `ibkr_client.place_order`.

In [5]:
config_no1 = {"configurable": {"thread_id": "notebook-graph-abort-1"}}
graph.invoke(buy_call("NVDA", 5), config=config_no1)
result = graph.invoke(Command(resume="no"), config=config_no1)
print(result["messages"][-1].content)

Order cancelled — first confirmation was not a clear yes.


In [6]:
config_no2 = {"configurable": {"thread_id": "notebook-graph-abort-2"}}
graph.invoke(buy_call("NVDA", 5), config=config_no2)
graph.invoke(Command(resume="yes"), config=config_no2)  # pass confirmation #1
result = graph.invoke(Command(resume="no"), config=config_no2)  # fail confirmation #2
print(result["messages"][-1].content)

Order cancelled at final confirmation — nothing was submitted.


### 1b. Orders over buying power are rejected before any confirmation

This check runs *before* the first `interrupt()`, so an oversized order
never even prompts the human — it just fails fast.

In [7]:
config_big = {"configurable": {"thread_id": "notebook-graph-toolarge"}}
result = graph.invoke(buy_call("AAPL", 100_000), config=config_big)
print("Interrupted:", "__interrupt__" in result)
print(result["messages"][-1].content)

Interrupted: False
Cannot place order: estimated cost $19,410,000.00 for 100000.0 AAPL @ ~$194.10 exceeds available buying power of $82,400.00.


## 2. Full agent, real LLM — happy path

`app/agent.py`'s `ask()` transparently detects a paused thread and resumes
it via `Command(resume=...)` instead of starting a new turn. From the
caller's side (CLI, FastAPI, or here) it's just "keep calling `ask()` with
whatever the user typed next." Requires `GROQ_API_KEY` in `.env`.

In [8]:
import re
import time

from groq import RateLimitError

from app.agent import ask


def ask_retry(question: str, thread_id: str, max_retries: int = 8) -> str:
    """ask() wrapped with rate-limit backoff. The shared Groq free tier
    used for this demo caps at 8000 tokens/minute, which a notebook that
    fires several tool-schema-heavy calls back to back can hit easily —
    this just waits out whatever cooldown Groq reports and retries."""
    for attempt in range(max_retries):
        try:
            return ask(question, thread_id)
        except RateLimitError as e:
            wait = 5.0
            match = re.search(r"try again in ([\d.]+)s", str(e))
            if match:
                wait = float(match.group(1)) + 1.0
            print(f"Rate limited, waiting {wait:.1f}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait)
    raise RuntimeError("Exceeded retries waiting for Groq rate limit to clear")

In [9]:
thread_id = "notebook-agent-happy-path"
print(ask_retry("buy 10 shares of AAPL", thread_id))

Confirm order: BUY 10.0 AAPL @ ~$194.10 (estimated cost $1,941.00). Reply 'yes' to continue or 'no' to cancel.


In [10]:
print(ask_retry("yes", thread_id))

Final confirmation — this will submit a real order: BUY 10.0 AAPL @ ~$194.10 (estimated cost $1,941.00). This cannot be undone once submitted. Reply 'yes' to submit or 'no' to cancel.


In [11]:
print(ask_retry("yes", thread_id))

Your order to buy **10 shares of AAPL** has been submitted (order ID 2002). I’ll let you know once it’s filled or if there are any updates.

Is there anything else you’d like to check—such as your current positions, account summary, open orders, or research on another ticker?


## 3. Full agent, real LLM — decline and resume normal conversation

Declining at confirmation #1 should cancel cleanly, and the same thread
should go right back to answering ordinary read-only questions afterward —
proof the interrupt doesn't leave the conversation stuck.

In [12]:
thread_id_decline = "notebook-agent-decline"
print(ask_retry("buy 5 shares of NVDA", thread_id_decline))

Confirm order: BUY 5.0 NVDA @ ~$131.20 (estimated cost $656.00). Reply 'yes' to continue or 'no' to cancel.


In [13]:
print(ask_retry("no", thread_id_decline))

I wasn’t able to place the order because the required confirmation wasn’t received, so the request was cancelled.

If you’d still like to buy 5 shares of NVDA, just let me know and we can start the purchase process again. If you’d prefer to do something else—review your portfolio, get a research summary on NVDA, or check open orders—just tell me what you’d like to see.


In [14]:
print(ask_retry("what are my current positions?", thread_id_decline))

Here’s a snapshot of your current holdings:

| Symbol | Shares | Avg. Cost | Current Price | Unrealized P&L | Market Value |
|--------|--------|----------|---------------|----------------|--------------|
| **AAPL** | 150.0 | $187.32 | $194.10 | **+3.6 %** | $29,115.00 |
| **NVDA** | 40.0 | $118.50 | $131.20 | **+10.7 %** | $5,248.00 |
| **MSFT** | 60.0 | $402.10 | $398.55 | **‑0.9 %** | $23,913.00 |

Let me know if you’d like to:

* See a detailed account summary (cash, buying power, etc.)  
* Review any open orders or recent fills  
* Get a research summary on any of these tickers (or another stock)  
* Place a new trade (buy)  

Just tell me what you’d like to explore next!


## 4. Full agent, real LLM — the model can't dodge the gate with a vague ask

Asking "should I buy TSLA" is a research question, not a buy instruction —
the system prompt tells the model to route this to `research_stock`, not
`buy_stock`. This confirms that boundary holds in practice, not just in
the prompt text.

In [15]:
thread_id_vague = "notebook-agent-vague"
print(ask_retry("should I buy TSLA?", thread_id_vague))

Here’s a concise overview of Tesla (TSLA) based on the latest data:

**Fundamentals**  
- 2023 revenue was about $81 billion (+ 15 % YoY) with net income of roughly $12.5 billion, marking a shift from loss‑making growth to consistent profitability.  
- Automotive gross margins sit in the high‑teens (≈ 18 %) and operating margins are in the low‑teens, helped by higher‑priced models, scale efficiencies, and growing software‑as‑a‑service revenue.  
- Valuation remains lofty: forward P/E is near 60×, reflecting strong market expectations for continued EV adoption and expansion of Tesla’s energy‑tech and services businesses.

**Technicals**  
- Since early 2024 the stock has traded in a relatively flat‑to‑slightly‑upward range, bouncing between a support zone around the low‑to‑mid $150 range and resistance near $190‑$200.  
- Volatility is higher than the broader market, with wide daily price swings and elevated implied‑volatility on options, indicating that price moves tend to be larger th

## 5. Inspect raw graph state on a paused thread

Handy for debugging: `agent.get_state(config).interrupts` is exactly what
`ask()` checks to decide whether the next call is a resume or a fresh
question.

In [16]:
from app.agent import agent

thread_id_inspect = "notebook-agent-inspect"
ask_retry("buy 3 shares of MSFT", thread_id_inspect)

config = {"configurable": {"thread_id": thread_id_inspect}}
state = agent.get_state(config)
print("Pending interrupts:", state.interrupts)

Pending interrupts: (Interrupt(value={'type': 'confirm_buy_order', 'step': 1, 'of': 2, 'message': "Confirm order: BUY 3.0 MSFT @ ~$398.55 (estimated cost $1,195.65). Reply 'yes' to continue or 'no' to cancel.", 'symbol': 'MSFT', 'quantity': 3.0, 'estimated_price': 398.55, 'estimated_cost': 1195.65}, id='703901d86f848c17749c3398ad1e3c47'),)
